In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

llm = ChatOpenAI()

In [4]:
class JokeState(TypedDict):

    topic : str
    joke : str
    explaination : str

In [5]:
def generate_joke(state : JokeState):

    prompt = f'generate a joke on the topic {state['topic']}'
    response = llm.invoke(prompt).content

    return {'joke' : response}

In [6]:
def generate_explaination(state : JokeState):

    prompt =  f'generate an explaination for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explaination' : response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {'configurable' : {'thread_id' : '1'}}
workflow.invoke({'topic' : 'pizza'}, config=config1)


{'topic': 'pizza',
 'joke': "Why couldn't the pizza make it to the party?\n\nBecause it was too cheesy to handle the delivery!",
 'explaination': 'This joke plays on the double meaning of "cheesy." In one sense, "cheesy" can refer to the literal cheese on a pizza, implying that there was too much cheese on the pizza for it to be able to make it to the party. In another sense, "cheesy" can also mean something that is overly sentimental or corny, suggesting that the pizza couldn\'t handle the emotional weight of the delivery. These multiple interpretations add humor to the joke.'}

In [13]:
workflow.invoke({'topic' : 'ralway food'}, config=config1)

{'topic': 'ralway food',
 'joke': "Why don't trains serve hot dogs? Because they don't want any passengers to go off the rails!",
 'explaination': 'This joke plays on the double meaning of the phrase "go off the rails." In the literal sense, it means for a train to physically go off the tracks. However, in this joke, it is used figuratively to mean for someone to act irrationally or lose control. The punchline suggests that if trains served hot dogs, it might lead to passengers acting erratically or "going off the rails," causing chaos on the train.'}

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why couldn't the pizza make it to the party?\n\nBecause it was too cheesy to handle the delivery!", 'explaination': 'This joke plays on the double meaning of "cheesy." In one sense, "cheesy" can refer to the literal cheese on a pizza, implying that there was too much cheese on the pizza for it to be able to make it to the party. In another sense, "cheesy" can also mean something that is overly sentimental or corny, suggesting that the pizza couldn\'t handle the emotional weight of the delivery. These multiple interpretations add humor to the joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1372-418e-6609-8002-453f3bbb8591'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-15T18:56:21.212489+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1372-2b58-651c-8001-c7581ab05344'}}, tasks=(), interrupts=()),
 St

In [15]:
config2 = {'configurable' : {'thread_id' : '2'}}
workflow.invoke({'topic' : 'corruption'}, config=config2)

{'topic': 'corruption',
 'joke': 'Why did the corrupt politician bring a ladder to work? \nTo climb their way up the corporate ladder of corruption!',
 'explaination': 'This joke plays on the idea of a "corporate ladder" as a metaphor for advancing in one\'s career. In this case, the corrupt politician is using a physical ladder to symbolize their willingness to climb to the top through corrupt practices and dishonest actions. It highlights the idea that corrupt individuals will do whatever it takes to advance their own interests, even if it means breaking the rules.'}

In [17]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'corruption', 'joke': 'Why did the corrupt politician bring a ladder to work? \nTo climb their way up the corporate ladder of corruption!', 'explaination': 'This joke plays on the idea of a "corporate ladder" as a metaphor for advancing in one\'s career. In this case, the corrupt politician is using a physical ladder to symbolize their willingness to climb to the top through corrupt practices and dishonest actions. It highlights the idea that corrupt individuals will do whatever it takes to advance their own interests, even if it means breaking the rules.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1234-e1f0-68f9-8002-1cf6987d5fbc'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-15T16:34:21.782429+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1234-c862-67e0-8001-d1cc7de29c01'}}, tasks=(), interrupts=()),
 StateSnapsho

## Time Travel

In [10]:
workflow.get_state({"configurable" : {"thread_id" : "1","checkpoint_id" : '1f1b1372-060b-61ce-8000-4f0a0c469a7c'} })

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b1372-060b-61ce-8000-4f0a0c469a7c'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-15T18:56:14.972136+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1372-0607-6fbe-bfff-8231a35cda01'}}, tasks=(PregelTask(id='51b09f60-e92f-a84a-2c5c-8d44b6273319', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Why couldn't the pizza make it to the party?\n\nBecause it was too cheesy to handle the delivery!"}),), interrupts=())

In [11]:
workflow.invoke(None,{"configurable" : {"thread_id" : "1","checkpoint_id" : '1f1b1372-060b-61ce-8000-4f0a0c469a7c'}})

{'topic': 'pizza',
 'joke': 'What did the pizza say to the delivery driver? "You wanna pizza me? Let\'s make a slice deal!"',
 'explaination': 'This joke plays on the double meaning of the word "pizza." In the context of the joke, "pizza" is used as both a noun referring to the food item and a verb meaning "to peace" or "to cease." The pizza is jokingly challenging the delivery driver to a confrontation by asking if they want to "pizza" (peace) them, but then offers to make a "slice" (peace) deal instead. The pun on "slice deal" also references the slices of pizza being delivered by the driver.'}

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What did the pizza say to the delivery driver? "You wanna pizza me? Let\'s make a slice deal!"', 'explaination': 'This joke plays on the double meaning of the word "pizza." In the context of the joke, "pizza" is used as both a noun referring to the food item and a verb meaning "to peace" or "to cease." The pizza is jokingly challenging the delivery driver to a confrontation by asking if they want to "pizza" (peace) them, but then offers to make a "slice" (peace) deal instead. The pun on "slice deal" also references the slices of pizza being delivered by the driver.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1378-a4ce-6dd9-8003-21170ddcb3f0'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-15T18:59:12.681087+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1378-8bd3-6f14-8002-2f2464c1a1d4'}}, tasks=(), i

## Update State

In [13]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b1372-060b-61ce-8000-4f0a0c469a7c", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b1381-c084-62ca-8001-4127c06fb2b4'}}

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1381-c084-62ca-8001-4127c06fb2b4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-09-15T19:03:17.178414+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1372-060b-61ce-8000-4f0a0c469a7c'}}, tasks=(PregelTask(id='96964b17-83f1-7232-9f8c-0c467689ec38', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'What did the pizza say to the delivery driver? "You wanna pizza me? Let\'s make a slice deal!"', 'explaination': 'This joke plays on the double meaning of the word "pizza." In the context of the joke, "pizza" is used as both a noun referring to the food item and a verb meaning "to peace" or "to cease." The pizza is jokingly chal

In [15]:
workflow.invoke(None,{"configurable" : {"thread_id" : "1","checkpoint_id" : '1f1b1372-060b-61ce-8000-4f0a0c469a7c'}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor?\nBecause it was feeling a little crusty!',
 'explaination': 'This joke plays on the double meaning of the word "crusty." On one hand, crusty can refer to the outer layer of a pizza, which is typically made of dough. On the other hand, crusty can also be used to describe someone or something that is irritable or grumpy. In this case, the pizza went to the doctor because it was feeling "crusty" in the sense of being physically unwell, but the punchline reveals that it was actually feeling "crusty" in the sense of having a crusty outer layer.'}

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\nBecause it was feeling a little crusty!', 'explaination': 'This joke plays on the double meaning of the word "crusty." On one hand, crusty can refer to the outer layer of a pizza, which is typically made of dough. On the other hand, crusty can also be used to describe someone or something that is irritable or grumpy. In this case, the pizza went to the doctor because it was feeling "crusty" in the sense of being physically unwell, but the punchline reveals that it was actually feeling "crusty" in the sense of having a crusty outer layer.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1388-d53e-62dd-8003-4900ce46fb0e'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-15T19:06:27.256562+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1388-bdaf-6aa0-8002-9a82356ae8a7'}}, ta

In [17]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1b1388-a724-61be-8001-6ce9ff1e1a5b", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b138b-80dc-6ece-8002-ec6606e5a439'}}

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b138b-80dc-6ece-8002-ec6606e5a439'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-09-15T19:07:38.939333+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1388-a724-61be-8001-6ce9ff1e1a5b'}}, tasks=(PregelTask(id='774ab408-1235-e486-047d-c15369013ccd', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor?\nBecause it was feeling a little crusty!', 'explaination': 'This joke plays on the double meaning of the word "crusty." On one hand, crusty can refer to the outer layer of a pizza, which is typically made of dough. On the other hand, crusty can also be used to describe someone or something tha

In [19]:
workflow.invoke(None,{"configurable" : {"thread_id" : "1","checkpoint_id" : '1f1b138b-80dc-6ece-8002-ec6606e5a439'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy? Because it had too many layers to unpack!',
 'explaination': 'This joke plays on the idea of the samosa, a popular Indian snack that features multiple layers of pastry stuffed with savory fillings. The punchline of the joke, "Because it had too many layers to unpack", is a play on words that refers to the samosa\'s literal layers as well as the emotional complexity of the samosa seeking therapy to work through its issues. Overall, the joke is a light-hearted way to compare the layers of a samosa to the layers of emotions that one may need to unpack in therapy.'}